# Version finale - Embeddings calculés à la volée

In [ ]:
import pandas as pd 
import numpy as np

language = "english" # or "french"
language_short = language[:2] 
df = pd.read_csv("./theses-soutenues-curated-stratified.csv")
docs = np.array(df[f"resumes.{language_short}"]) # 6500 rows
model_name = "Alibaba-NLP/gte-multilingual-base"

from simplemma import lemmatize
from string import punctuation
from sklearn.feature_extraction.text import CountVectorizer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
RANDOM_SEED = 2306406
np.random.seed(RANDOM_SEED)

class CustomLemmatizer:
    """An object to apply the lemmatize function."""

    def __init__(self, language, stop_words) -> None:
        self.__language = (language,)
        self.__stop_words = stop_words

    def __call__(self, doc: str) -> list[str]:
        """ """
        doc = doc.lower()
        doc = "".join([c for c in doc if c not in punctuation])
        out = []
        for word in doc.split(" "):
            if (word not in self.__stop_words) and (len(word) > 0):
                try:
                    lemma = lemmatize(word, lang=self.__language)
                except Exception as e:
                    # if lemmatization failed, skip it and use the word instead
                    lemma = word
                    print(f"BERTopic - Lemmatization failed (word: {word}) - Error : {e}")
                if lemma not in self.__stop_words:
                    out += [lemma]
        return out
    

vectorizer_model = CountVectorizer(
    tokenizer=CustomLemmatizer(language_short, list(stopwords(language_short)))
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30, 
    min_samples=10,
    prediction_data=True
)

umap_model = UMAP(
    n_neighbors = 50,
    metric = "cosine",
    n_components = 8,
    min_dist=0.0,
    low_memory = False,
    random_state=RANDOM_SEED   
)

topic_model = BERTopic(
	language = language,
    embedding_model = "Alibaba-NLP/gte-multilingual-base",
	vectorizer_model = vectorizer_model,
    umap_model= umap_model,
    hdbscan_model=hdbscan_model
)
topics, probabilities = topic_model.fit_transform(
    documents=docs, 
)

umap_model_for_vis = UMAP(
    n_components = 2,
    
    n_neighbors = 50,
    metric = "cosine",
    min_dist=0.0,
    low_memory = False,
    random_state=RANDOM_SEED   
)

reduced_embeddings = umap_model_for_vis.fit_transform(embeddings)

topic_model.reduce_topics(docs=docs, nr_topics=10)

topic_model.visualize_documents(
    docs = docs,
    # topics = reduced_topics, 
    reduced_embeddings = reduced_embeddings,
    hide_annotations = True, 
    hide_document_hover = True,
    height = 600, 
    width = 1000
)

# Version finale - Embeddings pré calculés

In [ ]:
from datasets import load_from_disk
import numpy as np

language = "english" # or "french"
language_short = language[:2] 
ds = load_from_disk(f"./embeddings/gte-multilingual-base-{language_short}-SBERT")
docs = np.array(ds[f"resumes.{language_short}"]) # 6500 rows
embeddings = np.array(ds["embedding"])			 # Shape : 6500 x 768

from simplemma import lemmatize
from string import punctuation
from sklearn.feature_extraction.text import CountVectorizer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
RANDOM_SEED = 2306406
np.random.seed(RANDOM_SEED)

class CustomLemmatizer:
    """An object to apply the lemmatize function."""

    def __init__(self, language, stop_words) -> None:
        self.__language = (language,)
        self.__stop_words = stop_words

    def __call__(self, doc: str) -> list[str]:
        """ """
        doc = doc.lower()
        doc = "".join([c for c in doc if c not in punctuation])
        out = []
        for word in doc.split(" "):
            if (word not in self.__stop_words) and (len(word) > 0):
                try:
                    lemma = lemmatize(word, lang=self.__language)
                except Exception as e:
                    # if lemmatization failed, skip it and use the word instead
                    lemma = word
                    print(f"BERTopic - Lemmatization failed (word: {word}) - Error : {e}")
                if lemma not in self.__stop_words:
                    out += [lemma]
        return out
    

vectorizer_model = CountVectorizer(
    tokenizer=CustomLemmatizer(language_short, list(stopwords(language_short)))
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30, 
    min_samples=10,
    prediction_data=True
)

umap_model = UMAP(
    n_neighbors = 50,
    metric = "cosine",
    n_components = 8,
    min_dist=0.0,
    low_memory = False,
    random_state=RANDOM_SEED   
)

topic_model = BERTopic(
	language = language,
	vectorizer_model = vectorizer_model,
    umap_model= umap_model,
    hdbscan_model=hdbscan_model
)
topics, probabilities = topic_model.fit_transform(
    documents=docs, 
    embeddings=embeddings
)

umap_model_for_vis = UMAP(
    n_components = 2,
    
    n_neighbors = 50,
    metric = "cosine",
    min_dist=0.0,
    low_memory = False,
    random_state=RANDOM_SEED   
)

reduced_embeddings = umap_model_for_vis.fit_transform(embeddings)

topic_model.reduce_topics(docs=docs, nr_topics=10)

topic_model.visualize_documents(
    docs = docs,
    # topics = reduced_topics, 
    reduced_embeddings = reduced_embeddings,
    hide_annotations = True, 
    hide_document_hover = True,
    height = 600, 
    width = 1000
)